This project is made using many different tools and 2 API's. API used here are Google Gemini and Wikipedia. Tools used are TensorFlow/Keras-LSTM, Numpy, Regular Experssions and Google Colab.
Dataset used here are- Cornell Movie-Dialogs Corpus, Glove- Global Vectors for Word Representation, Wikipedia Knowledge Base, Gemini.

In [ ]:
# Install required libraries
!pip install -U -q google-generativeai
!pip install -q wikipedia

# Download GloVe Word Embeddings (needed for Cell 4)
import os
if not os.path.exists('glove.6B.zip'):
    print("Downloading GloVe Brain... (takes about 1 min)")
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip

print("Environment Setup Complete.")

  Preparing metadata (setup.py) ... done
Environment Setup Complete.


In [ ]:
import os
import google.generativeai as genai
from google.colab import userdata
import wikipedia, re, numpy as np

# 1. Setup Gemini (The "Thinking" Brain)
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-pro')
    print("Gemini API Linked successfully!")
except:
    print("Gemini Key not found in Colab Secrets. Check the 🔑 icon on the left.")

# 2. Download Cornell Movie Data
if not os.path.exists('cornell_movie_dialogs_corpus.zip'):
    print("Downloading Movie Dataset...")
    !wget -q http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip
    !unzip -q cornell_movie_dialogs_corpus.zip

print("Dataset Downloaded and Extracted.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


🚀 Gemini API Linked successfully!
Dataset Downloaded and Extracted.


In [ ]:
# Cell 3: Improved Data Loading
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Increase sample size for more "Movie Wisdom"
num_samples = 30000

def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r"i'm", "i am", text); text = re.sub(r"he's", "he is", text)
    text = re.sub(r"\'ll", " will", text); text = re.sub(r"can't", "cannot", text)
    text = re.sub(r"[-()\"#/@;:<>{}+=~|.?,]", "", text)
    return text

lines = open('cornell movie-dialogs corpus/movie_lines.txt', encoding='utf-8', errors='ignore').read().split('\n')
conv_lines = open('cornell movie-dialogs corpus/movie_conversations.txt', encoding='utf-8', errors='ignore').read().split('\n')
id2line = {line.split(' +++$+++ ')[0]: line.split(' +++$+++ ')[-1] for line in lines if len(line.split(' +++$+++ ')) == 5}

questions, answers = [], []
for line in conv_lines[:-1]:
    parts = line.split(' +++$+++ ')[-1][1:-1].replace("'", "").replace(" ", "").split(',')
    for i in range(len(parts) - 1):
        q, a = clean_text(id2line[parts[i]]), clean_text(id2line[parts[i+1]])
        if 3 <= len(q.split()) <= 12 and 3 <= len(a.split()) <= 12:
            questions.append(q)
            answers.append('startseq ' + a + ' endseq')

questions = questions[:num_samples]
answers = answers[:num_samples]

tokenizer = Tokenizer(filters='', lower=False)
tokenizer.fit_on_texts(questions + answers)
vocab_size = len(tokenizer.word_index) + 1
max_len = 15 # Increased for better context

quest_pad = pad_sequences(tokenizer.texts_to_sequences(questions), maxlen=max_len, padding='post')
ans_pad = pad_sequences(tokenizer.texts_to_sequences(answers), maxlen=max_len, padding='post')

decoder_input = ans_pad[:, :-1]
decoder_output = ans_pad[:, 1:]
print(f"Upgraded Cornell Dataset: {len(questions)} high-quality pairs.")

Upgraded Cornell Dataset: 30000 high-quality pairs.


In [ ]:
embeddings_index = {}
with open('glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        embeddings_index[values[0]] = np.asarray(values[1:], dtype='float32')

embedding_matrix = np.zeros((vocab_size, 100))
for word, i in tokenizer.word_index.items():
    vec = embeddings_index.get(word)
    if vec is not None: embedding_matrix[i] = vec

print("GloVe Embedding Matrix Created.")

GloVe Embedding Matrix Created.


In [ ]:
# Cell 5: Upgraded Bidirectional Architecture
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Bidirectional, Concatenate, Dropout

# --- ENCODER ---
encoder_inputs = Input(shape=(max_len,))
enc_emb = Embedding(vocab_size, 100, weights=[embedding_matrix], trainable=False)(encoder_inputs)

# Bidirectional LSTM (Captures context from both directions)
encoder_lstm = Bidirectional(LSTM(256, return_state=True, dropout=0.2))
encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_lstm(enc_emb)

# Concatenate states
state_h = Concatenate()([forward_h, backward_h])
state_c = Concatenate()([forward_c, backward_c])
encoder_states = [state_h, state_c]

# --- DECODER ---
decoder_inputs = Input(shape=(max_len-1,))
dec_emb_layer = Embedding(vocab_size, 100, weights=[embedding_matrix], trainable=False)
dec_emb = dec_emb_layer(decoder_inputs)

# Decoder LSTM (512 units to match 256x2 Bidirectional input)
decoder_lstm = LSTM(512, return_sequences=True, return_state=True, dropout=0.2)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(vocab_size, activation='softmax')
output = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 15)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 15, 100)   │  1,906,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 14)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ [(None, 512),     │    731,136 │ embedding[0][0]   │
│ (Bidirectional)     │ (None, 256),      │            │                   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 14, 100)   │  1,906,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 512)       │          0 │ bidirectional[0]… │
│ (Concatenate)       │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 512)       │          0 │ bidirectional[0]… │
│ (Concatenate)       │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 14, 512), │  1,255,424 │ embedding_1[0][0… │
│                     │ (None, 512),      │            │ concatenate[0][0… │
│                     │ (None, 512)]      │            │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 14, 19060) │  9,777,780 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 15,576,340 (59.42 MB)

 Trainable params: 11,764,340 (44.88 MB)

 Non-trainable params: 3,812,000 (14.54 MB)

In [ ]:
# Cell 6: Higher Training (Use GPU!)
# We train for 100 epochs now to master the 30,000 samples
model.fit([quest_pad, decoder_input], decoder_output, epochs=100, batch_size=128)

Epoch 1/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 50s 177ms/step - loss: 3.5616
Epoch 2/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 80s 182ms/step - loss: 3.0363
Epoch 3/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 44s 186ms/step - loss: 2.9042
Epoch 4/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 44s 187ms/step - loss: 2.8105
Epoch 5/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 44s 187ms/step - loss: 2.7314
Epoch 6/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 44s 188ms/step - loss: 2.6613
Epoch 7/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 44s 187ms/step - loss: 2.5958
Epoch 8/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 44s 188ms/step - loss: 2.5341
Epoch 9/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 44s 188ms/step - loss: 2.4758
Epoch 10/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 44s 188ms/step - loss: 2.4210
Epoch 11/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 44s 189ms/step - loss: 2.3669
Epoch 12/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 44s 188ms/step - loss: 2.3155
Epoch 13/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 44s 188ms/step - loss: 2.2657
Epoch 14/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 44s 188ms/step - loss: 2.2197
E

In [ ]:
import numpy as np
import re
import wikipedia

# 1. SETUP INFERENCE MODELS
encoder_model = Model(encoder_inputs, encoder_states)
d_h, d_c = Input(shape=(256,)), Input(shape=(256,))
dec_out_inf, s_h, s_c = decoder_lstm(dec_emb_layer(decoder_inputs), initial_state=[d_h, d_c])
decoder_model = Model([decoder_inputs] + [d_h, d_c], [decoder_dense(dec_out_inf)] + [s_h, s_c])

# 2. WIKIPEDIA LOGIC
def get_wiki_clean(text):
    query = re.sub(r"(what is|who is|define|search for)", "", text).strip()
    if query.lower() == 'ai': query = 'Artificial Intelligence'
    try:
        search_res = wikipedia.search(query)
        if not search_res: return None
        return f"{wikipedia.summary(search_res[0], sentences=2, auto_suggest=False)} (Source: Wikipedia)"
    except: return None

# 3. GEMINI LOGIC
def get_smart_chat(text):
    try:
        response = gemini_model.generate_content(f"Be a smart assistant. Answer briefly: {text}")
        return response.text.strip()
    except: return "My cloud brain is fuzzy right now."

# 4. LSTM MOVIE BRAIN LOGIC
def get_movie_flair(text):
    try:
        input_seq = pad_sequences(tokenizer.texts_to_sequences([clean_text(text)]), maxlen=max_len, padding='post')
        states = encoder_model.predict(input_seq, verbose=0)
        target = np.zeros((1,1)); target[0,0] = tokenizer.word_index['startseq']
        res = ''
        for _ in range(max_len):
            out, h, c = decoder_model.predict([target] + states, verbose=0)
            idx = np.argmax(out[0, -1, :])
            word = next((w for w, i in tokenizer.word_index.items() if i == idx), '')
            if word == 'endseq' or word == '': break
            res += " " + word
            target[0,0] = idx; states = [h, c]
        return res.strip()
    except: return None

# --- MASTER CHAT LOOP ---
def chat():
    print("==============================================")
    print("   STABLE HYBRID BOT BY GAURAV THAKUR READY!  ")
    print("==============================================")

    while True:
        user_in = input("You: ").lower().strip()
        if user_in in ['exit', 'quit', 'bye']:
            print("Bot: Fade to black. Goodbye!"); break
        if not user_in: continue

        # PRIORITY 1: CREATOR IDENTITY
        if any(key in user_in for key in ['who created you', 'who made you', 'who is gaurav thakur', 'developer']):
            print("Bot: I was created and developed by Gaurav Thakur. He is the director of my code!")
            continue

        # PRIORITY 2: FACTUAL QUESTIONS (Wikipedia)
        if any(trigger in user_in for trigger in ['what is', 'who is', 'define']):
            print("Bot: [Searching Knowledge Base...]")
            wiki = get_wiki_clean(user_in)
            if wiki: print(f"Bot: {wiki}"); continue
            else:
                print("Bot: [Wiki empty... Checking Cloud Brain...]")
                print(f"Bot: {get_smart_chat(user_in)}"); continue

        # PRIORITY 3: BASIC GREETINGS
        greetings = {'hi': 'Hello! Ready for the next scene?', 'hello': 'Hi! I am your Hybrid AI assistant.'}
        if user_in in greetings:
            print(f"Bot: {greetings[user_in]}"); continue

        # PRIORITY 4: COMPLEX CHAT (Gemini)
        if len(user_in.split()) > 3:
            print("Bot: [Thinking...]")
            print(f"Bot: {get_smart_chat(user_in)}"); continue

        # PRIORITY 5: CASUAL CHAT (LSTM fallback)
        flair = get_movie_flair(user_in)
        if flair: print(f"Bot: {flair}")
        else: print("Bot: I am listening...")

chat()

   STABLE HYBRID BOT BY GAURAV THAKUR READY!  
You: HI
Bot: Hello! Ready for the next scene?
You: How are you
Bot: I am listening...
You: who is elon musk
Bot: [Searching Knowledge Base...]
Bot: Elon Reeve Musk ( EE-lon; born June 28, 1971) is a businessman and entrepreneur known for his leadership of Tesla, SpaceX, X, and xAI. Musk has been the wealthiest person in the world since 2025; as of April 2026, Forbes estimates his net worth to be US$809 billion.
Born into a wealthy family in Pretoria, South Africa, Musk emigrated in 1989 to Canada; he has Canadian citizenship since his mother was born there. (Source: Wikipedia)
You: who is modi
Bot: [Searching Knowledge Base...]
Bot: Narendra Damodardas Modi (born 17 September 1950) is an Indian politician who has served as the  prime minister of India since 2014. Modi was the chief minister of Gujarat from 2001 to 2014 and is the member of parliament (MP) for Varanasi. (Source: Wikipedia)
You: what is ai
Bot: [Searching Knowledge Base...]
